# 00 · Data Preparation
**ISOM5240 Group Project — Oriental Horizon Hotel Business**
*Review Intelligence for Hotel Revenue Management*

This notebook builds the two labelled datasets used to fine-tune our models:

| Task | Labels | Used by |
|------|--------|---------|
| **Review sentiment** | negative / neutral / positive | Notebook 01 |
| **Guest segment** | business / leisure | Notebook 02 |

**Source:** *515K Hotel Reviews Data in Europe* (Booking.com reviews, Kaggle, public domain).
Each review has a reviewer score (2.5–10) and trip tags such as `Business trip`, `Leisure trip`.
These provide our labels, so no manual labelling is needed.

**Steps**
1. Download the raw data from Kaggle.
2. Clean the review text.
3. Create labels from the reviewer score (sentiment) and the trip tags (segment).
4. Draw a class-balanced sample and split it into train / validation / test sets (80 / 10 / 10).
5. Save the six CSV files and upload them to the Hugging Face Hub, so every other notebook and
   the Streamlit app use exactly the same data.

*A CPU runtime is enough for this notebook.*

## 1. Install and import libraries

In [ ]:
%pip install -q kagglehub pandas scikit-learn huggingface_hub

In [ ]:
import ast
import glob
import os

import pandas as pd
from huggingface_hub import HfApi, whoami
from sklearn.model_selection import train_test_split

## 2. Configuration
All settings are in this one cell.

In [ ]:
# Hugging Face account that stores the dataset and the fine-tuned models.
HF_USERNAME = "ludanzhang54"
DATASET_REPO = f"{HF_USERNAME}/oriental-horizon-hotel-reviews"

# Kaggle dataset handle (public, no Kaggle login needed).
KAGGLE_DATASET = "jiashenliu/515k-hotel-reviews-data-in-europe"

# Optional: path to a manually uploaded Hotel_Reviews.csv. Leave as None to download from Kaggle.
RAW_CSV_PATH = None

OUTPUT_DIR = "data"          # where the prepared CSV files are written
RANDOM_SEED = 42             # fixed seed -> the same sample and split every run
SAMPLES_PER_CLASS = 15000    # balanced sample size per label (e.g. 15,000 x 3 = 45,000 sentiment reviews)

# Minimum review length (in words). Very short reviews carry too little information.
MIN_WORDS_SENTIMENT = 5
MIN_WORDS_SEGMENT = 20       # the trip purpose is only visible in longer reviews

SENTIMENT_LABELS = ["negative", "neutral", "positive"]
SEGMENT_LABELS = ["business", "leisure"]

## 3. Download the raw data

In [ ]:
def find_raw_csv():
    """Return the path of Hotel_Reviews.csv, downloading it from Kaggle if needed."""
    if RAW_CSV_PATH:
        return RAW_CSV_PATH
    import kagglehub
    folder = kagglehub.dataset_download(KAGGLE_DATASET)
    csv_files = glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True)
    if not csv_files:
        raise FileNotFoundError(f"No CSV file found in {folder}")
    return csv_files[0]


raw_csv = find_raw_csv()
raw = pd.read_csv(raw_csv)
print(f"Loaded {len(raw):,} reviews from {raw_csv}")
print("Columns:", list(raw.columns))
raw[["Hotel_Name", "Positive_Review", "Negative_Review", "Reviewer_Score", "Tags"]].head(3)

## 4. Clean the review text
Booking.com stores the *liked* and *disliked* parts of each review separately and writes
the placeholders `No Positive` / `No Negative` when a part is empty. We remove the placeholders
and join both parts into one free-text review, which is what hotel staff see on any review site.

In [ ]:
PLACEHOLDERS = {"no positive", "no negative", "nan", ""}


def clean_part(text):
    """Strip one review part and drop Booking.com's empty-review placeholders."""
    text = str(text).strip()
    return "" if text.lower() in PLACEHOLDERS else text


def build_review_text(row):
    """Join the positive and negative parts into one review with single spaces."""
    parts = [clean_part(row["Positive_Review"]), clean_part(row["Negative_Review"])]
    return " ".join(" ".join(parts).split())


reviews = raw.copy()
reviews["text"] = reviews.apply(build_review_text, axis=1)
reviews["word_count"] = reviews["text"].str.split().str.len()
reviews = reviews[reviews["word_count"] > 0].drop_duplicates(subset="text")
print(f"{len(reviews):,} non-empty, unique reviews")
reviews["word_count"].describe()

## 5. Create the labels

**Sentiment** (from `Reviewer_Score`). We leave small gaps between the classes, because a
score of exactly 6 or 8.5 is ambiguous even for a human reader:

| Score | Label |
|-------|-------|
| below 5.5 | negative |
| 6.5 to 7.9 | neutral |
| 9.0 and above | positive |

**Guest segment** (from `Tags`). Business vs leisure is the basic segmentation in hotel
revenue management: business guests book weekdays and are less price-sensitive, while
leisure guests book weekends and holidays and compare prices.

| Tags | Label |
|------|-------|
| `Business trip` | business |
| `Leisure trip` | leisure |

*Model iteration:* our first version used three segments (business / couple / family) and
reached only 55.8% test accuracy, because most reviews do not say who the guest travelled
with (family was mistaken for couple 1 time in 3). Business vs leisure is easier to read from
the text and is the segmentation the pricing system needs.

In [ ]:
def sentiment_from_score(score):
    """Map a Booking.com reviewer score to a sentiment label (None = ambiguous gap)."""
    if score < 5.5:
        return "negative"
    if 6.5 <= score < 8.0:
        return "neutral"
    if score >= 9.0:
        return "positive"
    return None


def parse_tags(tag_string):
    """Turn the Tags column, e.g. "[' Leisure trip ', ' Couple ']", into a list of lower-case tags."""
    try:
        return [tag.strip().lower() for tag in ast.literal_eval(tag_string)]
    except (ValueError, SyntaxError):
        return []


def segment_from_tags(tags):
    """Map trip tags to a guest segment label (None = trip purpose not given)."""
    if "business trip" in tags:
        return "business"
    if "leisure trip" in tags:
        return "leisure"
    return None


reviews["sentiment"] = reviews["Reviewer_Score"].apply(sentiment_from_score)
reviews["segment"] = reviews["Tags"].apply(parse_tags).apply(segment_from_tags)

print("Sentiment label counts (before sampling):")
print(reviews["sentiment"].value_counts(dropna=False), "\n")
print("Segment label counts (before sampling):")
print(reviews["segment"].value_counts(dropna=False))

## 6. Balanced sampling and train / validation / test split
Positive reviews far outnumber negative ones. Training on the raw distribution would teach the
model to say "positive" almost every time. We therefore sample the **same number of reviews per
label**, then split 80 / 10 / 10 while keeping the label proportions equal in every split.

In [ ]:
def build_task_dataset(df, label_column, labels, min_words):
    """Filter, balance and split the reviews for one task. Returns (train, val, test)."""
    task = df[df[label_column].isin(labels) & (df["word_count"] >= min_words)]
    task = task[["text", label_column]].rename(columns={label_column: "label"})

    per_class = min(SAMPLES_PER_CLASS, task["label"].value_counts().min())
    balanced = (
        task.groupby("label", group_keys=False)
        .sample(n=per_class, random_state=RANDOM_SEED)
        .reset_index(drop=True)
    )

    train, rest = train_test_split(
        balanced, test_size=0.2, stratify=balanced["label"], random_state=RANDOM_SEED
    )
    val, test = train_test_split(
        rest, test_size=0.5, stratify=rest["label"], random_state=RANDOM_SEED
    )
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)


splits = {
    "sentiment": build_task_dataset(reviews, "sentiment", SENTIMENT_LABELS, MIN_WORDS_SENTIMENT),
    "segment": build_task_dataset(reviews, "segment", SEGMENT_LABELS, MIN_WORDS_SEGMENT),
}

## 7. Dataset summary (numbers for the project report)

In [ ]:
summary_rows = []
for task, (train, val, test) in splits.items():
    for split_name, split_df in [("train", train), ("validation", val), ("test", test)]:
        row = {"task": task, "split": split_name, "samples": len(split_df)}
        row.update(split_df["label"].value_counts().to_dict())
        row["avg_words"] = round(split_df["text"].str.split().str.len().mean(), 1)
        summary_rows.append(row)

label_columns = SENTIMENT_LABELS + SEGMENT_LABELS
dataset_summary = pd.DataFrame(summary_rows)
dataset_summary[label_columns] = dataset_summary[label_columns].fillna(0).astype(int)
dataset_summary = dataset_summary[["task", "split", "samples", *label_columns, "avg_words"]]
dataset_summary

In [ ]:
# A few examples of each label, to check that the labels make sense.
for task, (train, _, _) in splits.items():
    print(f"===== {task} =====")
    for label, group in train.groupby("label"):
        print(f"[{label}] {group['text'].iloc[0][:200]}")
    print()

## 8. Save the CSV files

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
for task, (train, val, test) in splits.items():
    train.to_csv(f"{OUTPUT_DIR}/{task}_train.csv", index=False)
    val.to_csv(f"{OUTPUT_DIR}/{task}_val.csv", index=False)
    test.to_csv(f"{OUTPUT_DIR}/{task}_test.csv", index=False)
dataset_summary.to_csv(f"{OUTPUT_DIR}/dataset_summary.csv", index=False)

dataset_card = f"""---
license: cc0-1.0
task_categories:
- text-classification
language:
- en
---
# Oriental Horizon hotel reviews (ISOM5240 group project)

Class-balanced samples of the public Kaggle dataset
[515K Hotel Reviews Data in Europe](https://www.kaggle.com/datasets/{KAGGLE_DATASET}).

* `sentiment_*.csv`: labels negative / neutral / positive, from the reviewer score
  (below 5.5 / 6.5 to 7.9 / 9.0 and above).
* `segment_*.csv`: labels business / leisure, from the trip tags.

Each task has up to {SAMPLES_PER_CLASS} reviews per label (see `dataset_summary.csv`), split 80/10/10 (seed {RANDOM_SEED}).
"""
with open(f"{OUTPUT_DIR}/README.md", "w") as f:
    f.write("\n".join(line.strip() for line in dataset_card.splitlines()))

print(sorted(os.listdir(OUTPUT_DIR)))

## 9. Upload to the Hugging Face Hub
Uploading needs a Hugging Face **write token** saved as the Colab secret `HF_TOKEN`
(🔑 icon in the left sidebar). Without a token this step is skipped, so the rest of
the notebook still runs.

In [ ]:
def hf_logged_in_user():
    """Return the logged-in Hugging Face username, or None when no token is available."""
    try:
        return whoami()["name"]
    except Exception:
        return None


if hf_logged_in_user():
    api = HfApi()
    api.create_repo(DATASET_REPO, repo_type="dataset", exist_ok=True)
    api.upload_folder(folder_path=OUTPUT_DIR, repo_id=DATASET_REPO, repo_type="dataset")
    print(f"Uploaded: https://huggingface.co/datasets/{DATASET_REPO}")
else:
    print("No Hugging Face token found - upload skipped. The CSV files are in the 'data' folder.")